# Disjoint dbPTM holdout v2 — amended cohort construction

This notebook applies the documented pre-inference v2 feasibility amendment. Version 1 retained 740 sites (369 negative, 371 positive) from 477 proteins but stopped 10 sites below its 750-site gate; no predictions were generated. Version 2 changes only the total minimum to 700, retains the 200-per-class and 400-protein gates, and verifies the exact v1 alignment artifacts and counts before freezing. The one-time frozen inference section was added only after the v2 cohort audit passed review.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Clone or update the private repository

Private-repository access uses a fine-grained read-only GitHub token stored in Colab Secrets as `GITHUB_TOKEN`.

In [ ]:
from pathlib import Path
import os, subprocess, sys
from google.colab import userdata

def secret(name):
    try:
        return userdata.get(name)
    except Exception:
        return None

GITHUB_TOKEN = secret('GITHUB_TOKEN')
git_env = os.environ.copy()
git_env['GIT_TERMINAL_PROMPT'] = '0'
askpass = Path('/tmp/ubipred_git_askpass.sh')
if GITHUB_TOKEN:
    askpass.write_text('#!/bin/sh\ncase "$1" in *Username*) echo x-access-token;; *) echo "$GITHUB_TOKEN";; esac\n')
    askpass.chmod(0o700)
    git_env.update({'GIT_ASKPASS': str(askpass), 'GITHUB_TOKEN': GITHUB_TOKEN})

REPO_ROOT = Path('/content/Ubiquitination-Sites-Prediction')
try:
    if not (REPO_ROOT / '.git').exists():
        subprocess.run(['git', 'clone', '--recurse-submodules', 'https://github.com/kaleabnega/Ubiquitination-Sites-Prediction.git', str(REPO_ROOT)], check=True, env=git_env)
    else:
        subprocess.run(['git', '-C', str(REPO_ROOT), 'pull', '--ff-only'], check=True, env=git_env)
        subprocess.run(['git', '-C', str(REPO_ROOT), 'submodule', 'update', '--init', '--recursive'], check=True, env=git_env)
finally:
    askpass.unlink(missing_ok=True)
os.chdir(REPO_ROOT)
subprocess.run(['git', 'rev-parse', 'HEAD'], check=True)

In [ ]:
%pip install -q -r experiment/requirements-colab.txt "tf-keras==2.19.0"
%pip uninstall -y torchao
!apt-get update -qq && apt-get install -y -qq mmseqs2

In [ ]:
import json

def run_live(command):
    command = [str(part) for part in command]
    print('$', ' '.join(command), flush=True)
    process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in process.stdout:
        print(line, end='', flush=True)
    code = process.wait()
    if code:
        raise subprocess.CalledProcessError(code, command)

## Frozen paths and source artifacts

In [ ]:
CONFIG = Path('experiment/configs/external_dbptm_disjoint_holdout_v2.json')
SOURCE_ROOT = Path('/content/drive/MyDrive/MMUbiPred-research/external/dbptm-disjoint-holdout-v1')
ROOT = Path('/content/drive/MyDrive/MMUbiPred-research/external/dbptm-disjoint-holdout-v2')
ARCHIVE = SOURCE_ROOT / 'source/Ubiquitination.tgz'
NORMALIZED_BENCHMARK = SOURCE_ROOT / 'source/dbptm_ubiquitination_normalized.csv'
SOURCE_AUDIT = SOURCE_ROOT / 'source/source_audit.json'
SEQUENCE_CACHE = SOURCE_ROOT / 'uniprot_sequences.json'
COHORT_DIR = ROOT / 'cohort'
V1_FEASIBILITY_DIR = SOURCE_ROOT / 'cohort'
TRAINING_SEQUENCE_CACHE = Path('/content/drive/MyDrive/MMUbiPred-research/context/uniprot_training_sequences.json')
CONTEXT_RUN_DIR = Path('/content/drive/MyDrive/MMUbiPred-research/experiments/mmubipred-context-only-final-v1-seed42')
PAPER_MODEL = Path('/content/drive/MyDrive/MMUbiPred-replication/artifacts/DeepUBI_AAindex_One_Hot_Emb_drop_out2423.h5')
PRIOR_BENCHMARK = Path('replication/MMUbiPred/benchmark.csv')
PRIOR_SEQUENCE_CACHE = Path('/content/drive/MyDrive/MMUbiPred-research/external/dbptm-ptmgpt2-v1/uniprot_external_sequences.json')
PRIOR_COHORT_DIR = Path('/content/drive/MyDrive/MMUbiPred-research/external/dbptm-ptmgpt2-v3/cohort')
ROOT.mkdir(parents=True, exist_ok=True)
for required in [CONFIG, ARCHIVE, NORMALIZED_BENCHMARK, SOURCE_AUDIT, SEQUENCE_CACHE, V1_FEASIBILITY_DIR / 'mmseqs_alignments.tsv', V1_FEASIBILITY_DIR / 'qualifying_homology_alignments.tsv', TRAINING_SEQUENCE_CACHE, CONTEXT_RUN_DIR / 'best.pt', CONTEXT_RUN_DIR / 'refit_manifest.json', CONTEXT_RUN_DIR / 'refit_summary.json', PAPER_MODEL, PRIOR_BENCHMARK, PRIOR_SEQUENCE_CACHE, PRIOR_COHORT_DIR / 'external_cohort.tsv', PRIOR_COHORT_DIR / 'cohort_lock.json']:
    if not required.exists():
        raise FileNotFoundError(required)
print('All frozen model, training, and prior-external artifacts were found.')

## Acquire and normalize the official benchmark

The v1 source archive, normalized benchmark, and audit are reused. Their checksums remain enforced, so no source data are changed by the amendment.

In [ ]:
run_live([
    sys.executable, '-u', 'experiment/scripts/normalize_dbptm_benchmark.py',
    '--archive', ARCHIVE,
    '--output', NORMALIZED_BENCHMARK,
    '--audit', SOURCE_AUDIT,
])
source_audit = json.loads(SOURCE_AUDIT.read_text())
display(source_audit)

## Resolve full proteins through UniProt

This stage reuses the completed v1 sequence cache. It is resumable, with zero missing identifiers expected.

In [ ]:
run_live([
    sys.executable, '-u', 'experiment/scripts/fetch_external_benchmark_sequences.py',
    '--benchmark', NORMALIZED_BENCHMARK,
    '--output', SEQUENCE_CACHE,
])

## Construct the disjoint, homology-filtered cohort

This repeats the unchanged leakage and homology filters in a new v2 output directory. It must reproduce the preserved v1 alignments and prospective 740-site cohort before the amended gate can freeze it. It writes no model predictions.

In [ ]:
LOCK_PATH = COHORT_DIR / 'cohort_lock.json'
FEASIBILITY_PATH = COHORT_DIR / 'cohort_feasibility.json'
if LOCK_PATH.exists():
    print('Existing frozen cohort found; construction was not repeated.')
else:
    try:
        run_live([
            sys.executable, '-u', 'experiment/scripts/prepare_external_benchmark.py',
            '--config', CONFIG,
            '--benchmark', NORMALIZED_BENCHMARK,
            '--external-sequence-cache', SEQUENCE_CACHE,
            '--training-sequence-cache', TRAINING_SEQUENCE_CACHE,
            '--released-model', PAPER_MODEL,
            '--context-run-dir', CONTEXT_RUN_DIR,
            '--prior-external-benchmark', PRIOR_BENCHMARK,
            '--prior-external-sequence-cache', PRIOR_SEQUENCE_CACHE,
            '--prior-external-cohort-dir', PRIOR_COHORT_DIR,
            '--feasibility-reference-dir', V1_FEASIBILITY_DIR,
            '--output-dir', COHORT_DIR,
        ])
    except subprocess.CalledProcessError:
        if FEASIBILITY_PATH.exists():
            print('Cohort was not frozen. Pre-inference feasibility audit:')
            display(json.loads(FEASIBILITY_PATH.read_text()))
        raise

cohort_lock = json.loads(LOCK_PATH.read_text())
display({
    'status': cohort_lock['status'],
    'validation': cohort_lock['validation'],
    'exact_leakage_filter': cohort_lock['exact_leakage_filter'],
    'homology_filter': cohort_lock['homology_filter'],
    'final_cohort': cohort_lock['final_cohort'],
})

## Cohort review completed

The v2 cohort was reviewed before this section was added: 740 sites (369 negative, 371 positive) across 477 canonical proteins. The preserved v1 alignment content and counts reproduced, and no prediction had been generated. The cells below now perform the single frozen evaluation; they do not train or tune either model.

## One-time frozen external inference and predeclared analysis

A GPU runtime is required for frozen context-model inference. Exact MMUbiPred runs on CPU in the same process. The primary comparison is context minus exact MMUbiPred AUROC with 10,000 whole-protein bootstrap replicates. Existing completed results are never overwritten.

In [ ]:
import torch

EVALUATION_DIR = ROOT / 'evaluation'
RESULTS_PATH = EVALUATION_DIR / 'external_test_metrics.json'
if RESULTS_PATH.exists():
    print('Existing completed frozen evaluation found; inference was not repeated.')
else:
    if not torch.cuda.is_available():
        raise RuntimeError('A GPU runtime is required for the one-time context inference.')
    run_live([
        sys.executable, '-u', 'experiment/scripts/evaluate_external_benchmark.py',
        '--config', CONFIG,
        '--cohort-dir', COHORT_DIR,
        '--model-file', PAPER_MODEL,
        '--context-run-dir', CONTEXT_RUN_DIR,
        '--output-dir', EVALUATION_DIR,
        '--batch-size', '32',
        '--num-workers', '0',
        '--allow-external-test',
    ])

In [ ]:
external_result = json.loads(RESULTS_PATH.read_text())
display({
    'comparison_valid': external_result['comparison_valid'],
    'confirmatory_primary_claim_supported': external_result['confirmatory_primary_claim_supported'],
    'cohort': external_result['cohort'],
    'primary_hypothesis': external_result['primary_hypothesis'],
})
display(external_result['exact_released_mmubipred'])
display(external_result['frozen_context_model'])
display(external_result['primary_candidate_minus_exact_mmubipred'])
display(external_result['paired_primary_candidate_vs_exact_mmubipred']['bootstrap']['metrics'])

## Post-primary exploratory 50/50 hybrid

This analysis was added after the primary context-versus-MMUbiPred result was inspected. It averages the already-saved exact-MMUbiPred and context probabilities with fixed weights `0.5/0.5`; it performs no model inference and no weight or threshold search. Results are exploratory and cannot rescue the failed confirmatory primary claim.

In [ ]:
SECONDARY_PATH = EVALUATION_DIR / 'external_secondary_comparisons.json'
if SECONDARY_PATH.exists():
    print('Existing saved-prediction hybrid analysis found; it was not repeated.')
else:
    run_live([
        sys.executable, '-u', 'experiment/scripts/analyze_external_saved_predictions.py',
        '--config', CONFIG,
        '--cohort-dir', COHORT_DIR,
        '--evaluation-dir', EVALUATION_DIR,
    ])

hybrid_result = json.loads(SECONDARY_PATH.read_text())
display({
    'analysis_role': hybrid_result['analysis_role'],
    'post_primary_exploratory': hybrid_result['post_primary_exploratory'],
    'confirmatory_claim_allowed': hybrid_result['confirmatory_claim_allowed'],
    'hybrid_architecture': hybrid_result['hybrid_architecture'],
})
display(hybrid_result['equal_probability_hybrid'])
display(hybrid_result['hybrid_minus_exact_mmubipred'])
display(hybrid_result['paired_hybrid_vs_exact_mmubipred']['bootstrap']['metrics'])
display(hybrid_result['paired_hybrid_vs_exact_mmubipred']['mcnemar_accuracy_descriptive_site_level'])